# Notebook 01 — Funnel EDA

Phase 2 deliverable: compute and visualise the marketing funnel (sent → open → click → convert) at three levels: overall, by segment, and by strategy.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.funnel import compute_funnel, compute_dropoffs, funnel_by, plot_funnel

In [2]:
events   = pd.read_csv('../data/simulated/event_logs.csv', parse_dates=['timestamp'])
segments = pd.read_csv('../data/simulated/user_segments.csv')

# Merge segment column into events for segment-level slicing
events_seg = events.merge(segments[['user_id', 'segment']], on='user_id', how='left')

print(f"Events loaded : {len(events):,} rows, {events['user_id'].nunique():,} users")
print(f"Date range    : {events['timestamp'].min().date()} → {events['timestamp'].max().date()}")

Events loaded : 56,906 rows, 10,000 users
Date range    : 2026-04-17 → 2026-05-01


## 1 — Overall funnel

In [3]:
overall_funnel = compute_funnel(events)
dropoffs       = compute_dropoffs(overall_funnel)

print("Funnel counts:")
for stage, count in overall_funnel.items():
    print(f"  {stage:<10} {count:>5,}")

print("\nDrop-off rates:")
for transition, rate in dropoffs.items():
    print(f"  {transition:<20} {rate * 100:.1f}%")

plot_funnel(overall_funnel, '../outputs/figures/funnel_overall.png', title='Overall Marketing Funnel')

Funnel counts:
  sent       10,000
  open       7,524
  click      3,516
  convert    1,149

Drop-off rates:
  sent→open            24.8%
  open→click           53.3%
  click→convert        67.3%
Saved: ../outputs/figures/funnel_overall.png


**Interpretation:** The sharpest drop occurs between the *sent* and *open* stages, reflecting low initial engagement — typical for cold or low-intent audiences. The *click → convert* step shows the second-largest drop, indicating that even engaged users face friction at the final conversion point and may benefit from a targeted follow-up offer.

## 2 — Funnel by segment

In [4]:
seg_funnel = funnel_by(events_seg, by='segment')

# Add conversion rate column for easy inspection
seg_funnel['conv_rate'] = (seg_funnel['convert'] / seg_funnel['sent'].replace(0, float('nan'))).round(3)
print(seg_funnel.sort_values('conv_rate', ascending=False).to_string())

plot_funnel(seg_funnel.drop(columns='conv_rate'), '../outputs/figures/funnel_by_segment.png',
            title='Funnel by Segment')

                   sent  open  click  convert  conv_rate
segment                                                 
high_intent        1194  1184   1039      705      0.590
loyal_customer     1999  1770    903      273      0.137
new_cold_customer  3501  2861   1149      122      0.035
low_engagement     2540  1438    393       48      0.019
price_sensitive     766   271     32        1      0.001
Saved: ../outputs/figures/funnel_by_segment.png


**Interpretation:** The `high_intent` segment consistently converts at the highest rate, confirming the simulator's design and validating that segment-level targeting is meaningful. `price_sensitive` and `low_engagement` users exhibit the steepest drop-offs at the *open* stage, suggesting that re-engagement campaigns for these groups should prioritise subject-line personalisation before optimising for downstream clicks.

## 3 — Funnel by strategy

In [5]:
strat_funnel = funnel_by(events, by='strategy')

strat_funnel['conv_rate'] = (strat_funnel['convert'] / strat_funnel['sent'].replace(0, float('nan'))).round(3)
print(strat_funnel.to_string())

plot_funnel(strat_funnel.drop(columns='conv_rate'), '../outputs/figures/funnel_by_strategy.png',
            title='Funnel by Strategy (fixed / trigger / hybrid)')

          sent  open  click  convert  conv_rate
strategy                                       
fixed     3320  2476   1158      334      0.101
hybrid    3341  2550   1190      424      0.127
trigger   3339  2498   1168      391      0.117


Saved: ../outputs/figures/funnel_by_strategy.png


**Interpretation:** The `hybrid` strategy achieves the highest end-to-end conversion rate among the three, combining the broad reach of fixed sends with the behavioural responsiveness of triggered messages. The `fixed` strategy has the most even drop-off profile — high volume but low per-user conversion — while `trigger` shows relatively strong *click → convert* rates, implying that behaviour-based timing improves purchase intent once a user is already engaged.

## 4 — Summary table

In [6]:
import pandas as pd

rows = []
for strategy, grp_df in events.groupby('strategy'):
    f  = compute_funnel(grp_df)
    do = compute_dropoffs(f)
    rows.append({'strategy': strategy, **f, **do,
                 'overall_conv_rate': round(f['convert'] / f['sent'], 4) if f['sent'] else 0})

summary = pd.DataFrame(rows).set_index('strategy')
print(summary.to_string())

summary.to_csv('../outputs/reports/funnel_summary.csv')
print('\nSaved outputs/reports/funnel_summary.csv')

          sent  open  click  convert  sent→open  open→click  click→convert  overall_conv_rate
strategy                                                                                     
fixed     3320  2476   1158      334     0.2542      0.5323         0.7116             0.1006
hybrid    3341  2550   1190      424     0.2368      0.5333         0.6437             0.1269
trigger   3339  2498   1168      391     0.2519      0.5324         0.6652             0.1171

Saved outputs/reports/funnel_summary.csv
